In [ ]:
import os
import datasets
from PIL import Image
import io

# -------------------------------------------------------
# Configuration
# -------------------------------------------------------
local_dir = r"OK VQA"
hdfs_dir = None  # Can set if needed
DATA_SOURCE = "HuggingFaceM4/A-OKVQA"

# -------------------------------------------------------
# Load dataset
# -------------------------------------------------------
print("Loading A-OKVQA dataset...")
dataset = datasets.load_dataset(DATA_SOURCE)
train_dataset = dataset["train"]
val_dataset = dataset["validation"]
#test_dataset = dataset["test"]

# -------------------------------------------------------
# Create train/val/test splits with reproducible random sampling
# -------------------------------------------------------
print("Creating reproducible splits...")

# Shuffle once with fixed seed
shuffled_train = train_dataset.shuffle(seed=42)
shuffled_val = val_dataset.shuffle(seed=42)
#shuffled_test = val_dataset.shuffle(seed=42)

train_dataset = shuffled_train.select(range(min(6000, len(shuffled_train))))
val_dataset = shuffled_val.select(range(min(100, len(shuffled_val))))
test_dataset = shuffled_val.select(range(100, min(600, len(shuffled_val))))

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

# -------------------------------------------------------
# Preprocessing function
# -------------------------------------------------------
def make_map_fn(split):
    def process_fn(example, idx):
        import io
        DATA_SOURCE = "HuggingFaceM4/A-OKVQA"  # define inside worker

        question = example.pop("question")
        options = example.pop("choices")
        answer_idx = example.pop("correct_choice_idx")
        image = example.pop("image") if "image" in example else None

        # Handle None or invalid answer_idx
        if answer_idx is None:
            # Skip this example by returning None or use a default
            print(f"Warning: Skipping example {idx} due to missing answer_idx")
            return None
        
        # Convert answer index to letter (A, B, C, ...)
        # Ensure answer_idx is an integer
        if isinstance(answer_idx, str):
            answer_idx = int(answer_idx)
        answer_letter = chr(65 + answer_idx)
        ground_truth = answer_letter

        # Format options as A) B) C) ...
        options_text = "\n".join([f"{chr(65+i)}) {opt}" for i, opt in enumerate(options)])

        image_link = f"aokvqa/{split}_image_{idx+1}"

        prompt_text = (
            r"<image> "
            + f"{question}\n"
            f"Options:\n{options_text}\n"
            f"The image_url is: {image_link}."
            + r" Solve this question step by step as instructed and choose the correct option and put the final answer inside <answer> \boxed{(option)answer} </answer> tags.\n "
        )

        # Convert image to bytes
        if image is not None:
            img_buffer = io.BytesIO()
            image.save(img_buffer, format="PNG")
            img_bytes = img_buffer.getvalue()
            images = [{"bytes": img_bytes, "path": None}]
        else:
            images = []

        data = {
            "data_source": DATA_SOURCE,
            "prompt": [
                {"role": "system",
                 "content":
                     r"""
You are a helpful multi-modal reasoning assistant specializing in math, science, and general knowledge questions.
You are given a question from an user and you have access to a set of Available_tools.
Answer in this sequence:
1. Start by thinking inside <think_reasoning> ... </think_reasoning> tags about the question and which tool to call if needed.

2. Then you can call tool or tools from Available_tools. Tool call instructions:
For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>{"name": <function-name>, "arguments": <args-json-object>}</tool_call> end of response.

List of Available_tools: captioning_tool, ocr_tool, detection_tool and perception_tool. (DO NOT make up any other tools or tool names. Only use the set of tools given to you. Also avoid repetition)

3. After you have used the tools, you will see the tool outputs inside appropriate tags in the same order from the system.

4. After getting tool_response, think Your thoughts on the tool_response inside <think_perception> ... </think_perception> tags once.(think_perception step only comes after tool_response tags)

5. Then resume your thought process inside <think_reasoning> ... </think_reasoning> tags again.
Try to think clearly, aloud and step-by-step so that you reach to the correct final answer. If needed also reflect on your thought process. All thinking must be inside <think_reasoning> ... </think_reasoning> tags.

6. At the end you must always choose the most appropriate option and put answer inside: <answer> \boxed{(option)answer} </answer> tags, irrespective of the output of the tool being true or false or incorrect.
You MUST provide an answer even if uncertain at the end.
"""
                },
                {"role": "user", "content": prompt_text},
            ],
            "images": images,
            "image_link": [image_link],
            "reward_model": {"style": "rule", "ground_truth": ground_truth},  # A/B/C/D
            "extra_info": {
                "split": split,
                "index": idx,
                "answer": ground_truth,  # letter
                "answer_idx": answer_idx,
                "question": "<image> " + question,
                "options": options,
                "tools_kwargs": {
                    "spectra_reward": {"create_kwargs": {"ground_truth": ground_truth}},
                },
            },
        }
        return data
    return process_fn

# -------------------------------------------------------
# Map preprocessing
# -------------------------------------------------------
print("Processing train dataset...")
train_dataset = train_dataset.map(function=make_map_fn("train"), with_indices=True, num_proc=4)
train_dataset = train_dataset.filter(lambda x: x is not None)

print("Processing val dataset...")
val_dataset = val_dataset.map(function=make_map_fn("val"), with_indices=True, num_proc=4)
val_dataset = val_dataset.filter(lambda x: x is not None)

print("Processing test dataset...")
test_dataset = test_dataset.map(function=make_map_fn("test"), with_indices=True, num_proc=4)
test_dataset = test_dataset.filter(lambda x: x is not None)

# -------------------------------------------------------
# Save to parquet
# -------------------------------------------------------
print("Saving datasets...")
os.makedirs(local_dir, exist_ok=True)
train_dataset.to_parquet(os.path.join(local_dir, "train_okvqa_6000.parquet"))
val_dataset.to_parquet(os.path.join(local_dir, "val_okvqa_100.parquet"))
test_dataset.to_parquet(os.path.join(local_dir, "test_okvqa_500.parquet"))

print(f"Datasets saved in {local_dir}")
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

Loading A-OKVQA dataset...
Creating reproducible splits...
Train size: 6000, Val size: 100, Test size: 500
Processing train dataset...
Processing val dataset...
Processing test dataset...
Saving datasets...


Creating parquet from Arrow format:   0%|          | 0/60 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Datasets saved in C:\Users\987ta\Desktop\Agentic Reasoning\2. Data\OK VQA
Train size: 6000, Val size: 100, Test size: 500
